# 第2回 宿題（発展）: 用語集の Q&A を LoRA で学習し、学習前後と検索方式を比べる

演習ノートブック（`notebooks/lora_format_tuning.ipynb`）をもとに、FAQ の代わりに **講座の用語集**（`data/glossary.json`）から作った Q&A を学習させます。

**確かめること**

1. 学習前のモデルは、講座の用語集の定義（講座に固有の内容を含む）を答えられるか。
2. 学習後、学習データと同じ言い回しの質問（`exact`）、言い換え・別名・逆引きの質問（`paraphrase`）、用語集にない語の質問（`unknown`）で、それぞれどう変わるか。
3. 学習しないまま用語集を検索してプロンプトに入れる方式（セクション 6）と比べて、どちらが向くか。

**データ**（`lectures/02/homework/` にあります）

| ファイル | 内容 |
| --- | --- |
| `glossary_sft.jsonl` | データ A。`uv run python scripts/make_sft_dataset.py` の出力そのまま（58 語 × 2 言い回し = 116 件） |
| `glossary_sft_plus.jsonl` | データ B。A に、用語集にない 15 語を聞かれたら「用語集に載っていません」と答える負例 30 件を足したもの（`make_plus_dataset.py` で作成） |
| `glossary_eval.json` | 評価セット 34 問（exact 12 / paraphrase 12 / unknown 10）と正解の語句 |
| `glossary.json` | 用語集（リポジトリの `data/glossary.json`）。セクション 6 の検索で使う |

**実行のしかた**

- **Colab**: T4 GPU を選び、上の 4 ファイルを左のファイル一覧から `/content` にアップロードしてから、上から実行します。
- **ローカル**（CUDA の GPU）: `lectures/02/homework/` に Colab と同じ版を入れた venv を作り、このディレクトリで実行します（`README.md` 参照）。
- どちらのデータで学習するかは、セクション 1 の `train_file` で選びます（環境変数 `GLOSSARY_TRAIN` でも変えられます）。A と B を比べるときは、**セッションを再起動して**から上から実行し直します（学習後のモデルが残っているため）。

## 0. 環境の確認とライブラリの導入

Colab のときだけ、演習と同じ版のライブラリを入れます。ローカルでは venv に入れた版をそのまま使います。

In [ ]:
import subprocess, sys
IN_COLAB = "google.colab" in sys.modules
try:
    gpu_info = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], capture_output=True, text=True).stdout
except FileNotFoundError:
    gpu_info = ""
print("Colab" if IN_COLAB else "ローカル", "|", gpu_info or "GPU が見つかりません")

In [ ]:
# 演習ノートブックと同じ版（Colab の T4 で動作確認済み）
PINS = {"transformers": "5.18.0", "peft": "0.21.1", "trl": "1.14.2", "accelerate": "1.15.0", "datasets": "5.1.0"}
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"])   # peft 0.21 と非互換なので外す
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *[f"{k}=={v}" for k, v in PINS.items()]])

import transformers, peft, trl, accelerate, datasets, torch
VERSIONS = {m.__name__: m.__version__ for m in [transformers, peft, trl, accelerate, datasets, torch]}
print(VERSIONS)
mismatch = {k: VERSIONS[k] for k, v in PINS.items() if VERSIONS[k] != v}
assert not mismatch, f"版が PINS と違います: {mismatch}"

## 1. 設定

演習からの変更点:

- `train_file`: 学習データのファイル（A: `glossary_sft.jsonl` / B: `glossary_sft_plus.jsonl`）。
- `num_train_epochs`: 20 → 10。データが 116 件（B は 146 件）あるので、1 エポックが約 15 ステップ（B は約 19 ステップ）になります。10 エポックで約 150 ステップです。
- `max_new_tokens`: 96 → 256。用語の定義文は FAQ の文より長いので、96 だと途中で切れます。
- `use_system_prompt`: 学習・評価の両方で、先頭に同じ system プロンプト（用語集アシスタントの役割と「用語集に載っていない語は推測で説明しない」という指示）を付けます。演習の `SYSTEM` と同じく、**指示はプロンプトで済ませてある**状態から始め、学習で何が変わるかを見ます。

In [ ]:
from pathlib import Path
import os, json, time

CONFIG = {
    "base_model": "Qwen/Qwen3.5-0.8B",
    "revision": "2fc06364715b967f1860aea9cf38778875588b17",
    "seed": 42,
    "train_file": os.environ.get("GLOSSARY_TRAIN", "glossary_sft.jsonl"),   # A: glossary_sft.jsonl / B: glossary_sft_plus.jsonl
    "num_train_epochs": 10,
    "warmup_steps": 10,
    "lr": 2e-4,
    "lora_r": 16,
    "lora_alpha": 32,
    "max_seq_len": 512,
    "max_new_tokens": 256,
    "gen": {"do_sample": False},
    "use_system_prompt": True,
}
VARIANT = Path(CONFIG["train_file"]).stem   # 結果ファイルの置き場所をデータごとに分ける
DATA_DIR = Path("/content") if IN_COLAB else Path.cwd()
GLOSSARY_FILE = DATA_DIR / "glossary.json"
if not GLOSSARY_FILE.exists():   # ローカルではリポジトリの data/glossary.json を使う
    GLOSSARY_FILE = Path.cwd().parents[2] / "data" / "glossary.json"
WORK = (Path("/content/glossary_lab") if IN_COLAB else Path.cwd() / "out") / VARIANT
WORK.mkdir(parents=True, exist_ok=True)
print(json.dumps(CONFIG, ensure_ascii=False, indent=2))
print("データ:", DATA_DIR, "| 用語集:", GLOSSARY_FILE, "| 結果の保存先:", WORK)

## 2. 学習データと評価セットの読み込み

学習データの各行は `{"messages": [user, assistant]}` の形で、演習の `to_chat` と同じ chat 形式です。

評価セットの 3 種類:

| kind | 例 | 正解の条件 |
| --- | --- | --- |
| `exact` | 「LoRA」とは何ですか？ | 学習データと同じ質問。定義文の語句（例: 「低ランク」と「行列」）を含む |
| `paraphrase` | Low-Rank Adaptationはどんな手法ですか？ / 学習データ全体を1周学習することを何と呼びますか？ | 学習データに無い言い回し、別名、逆引き（説明から用語名を答える）。語句を含む |
| `unknown` | 「RAG」とは何ですか？ / 「トークンブリッジ」とは何ですか？ | 用語集に無い語（実在する語 7、架空の語 3）。「用語集に載っていません」の旨を答える |

`must_include` は「グループごとにどれか 1 語」を含めば合格です。語句の一致は目安なので、出力の原文も自分の目で読んでください。

**見るところ**: `paraphrase` と `unknown` の質問が学習データに入っていないことを、ここで確かめます（入っているとエラーになります）。

In [ ]:
def read_jsonl(path):
    return [json.loads(line) for line in Path(path).read_text(encoding="utf-8").splitlines() if line.strip()]

train_rows = read_jsonl(DATA_DIR / CONFIG["train_file"])
EVAL_SET = json.loads((DATA_DIR / "glossary_eval.json").read_text(encoding="utf-8"))
EVAL_ITEMS, UNKNOWN_MARKERS = EVAL_SET["items"], EVAL_SET["unknown_markers"]

train_questions = {r["messages"][0]["content"] for r in train_rows}
leaked = [it["q"] for it in EVAL_ITEMS if it["kind"] != "exact" and it["q"] in train_questions]
assert not leaked, f"評価用の質問が学習データに入っています: {leaked}"
n_negative = sum("用語集に載っていません" in r["messages"][1]["content"] for r in train_rows)
print(f"学習データ {CONFIG['train_file']}: {len(train_rows)} 件（うち用語集にない語の負例 {n_negative} 件）")
print("評価セット:", {k: sum(it["kind"] == k for it in EVAL_ITEMS) for k in ["exact", "paraphrase", "unknown"]})
print("学習例の 1 件:", train_rows[0]["messages"])

## 3. モデルの読み込みと学習前評価

演習と同じく量子化なし（32bit）で読み込みます。生成は貪欲生成で、学習前・学習後・検索方式のすべてで同じ条件にします。

**採点のしかた**（以降のセクションでも同じ関数を使います）

- `exact` / `paraphrase`: 「用語集に載っていません」の旨を答えず、かつ `must_include` の語句を含めば ○。
- `unknown`: 「用語集に載っていません」の旨（`unknown_markers` のどれか）を答えれば ○。一般知識で説明しても、用語集にない語を説明した時点で ×（根拠のない断定と同じ扱い）。

**見るところ**: 種類ごとの正解率と、出力の原文。学習前のモデルは一般的な語（LoRA、トークンなど）を自分の知識で説明できても、講座に固有の内容（既定モデル、宿題の 3 レベルなど）は答えられないはずです。

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

DEVICE = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
tok = AutoTokenizer.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"])
if tok.pad_token is None: tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(CONFIG["base_model"], revision=CONFIG["revision"], dtype=torch.float32).to(DEVICE)
model.config.use_cache = True
print("loaded:", CONFIG["base_model"], "| device:", DEVICE, "| 重みのメモリ(GB):", round(model.get_memory_footprint() / 1e9, 2))

SYSTEM = ("あなたは AIE2026 講座の用語集アシスタントです。用語の意味を、講座の用語集の定義に沿って答えてください。"
          "用語集に載っていない語については、推測で説明せず『用語集に載っていません』と答えてください。")

def add_system(messages):
    return ([{"role": "system", "content": SYSTEM}] if CONFIG["use_system_prompt"] else []) + messages

def build_messages(question, context=None):
    # context: セクション 6 の検索方式で、用語集から取り出した項目をプロンプトに入れるときに使う
    user = question if context is None else f"用語集の該当項目:\n{context}\n\n質問: {question}"
    return add_system([{"role": "user", "content": user}])

def generate(model, question, context=None):
    prompt = tok.apply_chat_template(build_messages(question, context), tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inputs = tok(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=CONFIG["max_new_tokens"], pad_token_id=tok.pad_token_id, **CONFIG["gen"])
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

In [ ]:
import re
import pandas as pd
pd.set_option("display.max_colwidth", None)
KINDS = ["exact", "paraphrase", "unknown"]

def said_unknown(raw):
    return any(m in raw for m in UNKNOWN_MARKERS)

def judge(item, raw):
    if item["kind"] == "unknown":
        return said_unknown(raw)
    return not said_unknown(raw) and all(any(w in raw for w in group) for group in item["must_include"])

def mark(ok):
    return "○" if ok else "×"

def evaluate(model, label, retrieve=None):
    '''評価セット 34 問を生成して採点する。retrieve を渡すと、検索した用語集の項目をプロンプトに入れる'''
    rows, t0 = [], time.time()
    for it in EVAL_ITEMS:
        hits = retrieve(it["q"]) if retrieve else None
        context = None if hits is None else ("\n".join(f"- {t['term']}: {t['definition']}" for t in hits) or "（該当する項目はありません）")
        raw = generate(model, it["q"], context)
        row = {"kind": it["kind"], "q": it["q"], "raw": raw, "ok": judge(it, raw)}
        if hits is not None: row["retrieved"] = [t["id"] for t in hits]
        rows.append(row)
    summary = {"label": label, "n": len(rows), "overall": sum(r["ok"] for r in rows) / len(rows),
               **{k: sum(r["ok"] for r in rows if r["kind"] == k) / sum(r["kind"] == k for r in rows) for k in KINDS},
               "seconds": round(time.time() - t0, 1)}
    (WORK / f"eval_{label}.json").write_text(json.dumps({"summary": summary, "rows": rows}, ensure_ascii=False, indent=2), encoding="utf-8")
    print(json.dumps(summary, ensure_ascii=False))
    return summary, rows

before_summary, before_rows = evaluate(model, "before")
display(pd.DataFrame([{"kind": r["kind"], "質問": r["q"], "学習前の出力": r["raw"], "判定": mark(r["ok"])} for r in before_rows]))

## 4. LoRA 学習

演習のセクション 4 と同じ設定（LoRA を差し込む層、バッチ 4 × 勾配累積 2、cosine）で、データだけを用語集の Q&A に替えます。各行の `messages` の先頭に system プロンプトを足して、評価と同じ入力の形にそろえます。

**見るところ**

- `trainable%`: 演習と同じ割合になるはずです（データを替えても LoRA の大きさは変わらない）。
- `Training Loss`: 演習（14 件 × 20 周）より件数が多く、答えの文も長いので、下がり方を比べてください。
- `学習時間`: 演習の 40 ステップと比べて、ステップ数と時間がどれだけ増えたか。

In [ ]:
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
from datasets import Dataset

train_ds = Dataset.from_list([{"messages": add_system(r["messages"])} for r in train_rows])

model.config.use_cache = False
lora = LoraConfig(r=CONFIG["lora_r"], lora_alpha=CONFIG["lora_alpha"], lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
                  target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"])
sft = SFTConfig(output_dir=str(WORK / "ckpt"), num_train_epochs=CONFIG["num_train_epochs"], per_device_train_batch_size=4, gradient_accumulation_steps=2,
                learning_rate=CONFIG["lr"], lr_scheduler_type="cosine", warmup_steps=CONFIG["warmup_steps"], logging_steps=10, save_strategy="no",
                fp16=False, max_length=CONFIG["max_seq_len"], packing=False, report_to="none", seed=CONFIG["seed"])
trainer = SFTTrainer(model=model, args=sft, train_dataset=train_ds, peft_config=lora, processing_class=tok)
trainer.model.print_trainable_parameters()
t0 = time.time()
trainer.train()
TRAIN_SECONDS = round(time.time() - t0, 1)
LOSSES = [(h["step"], h["loss"]) for h in trainer.state.log_history if "loss" in h]
print(f"学習時間: {TRAIN_SECONDS:.0f} 秒 | ステップ数: {trainer.state.global_step} | エポック: {trainer.state.epoch:.1f} | loss: {LOSSES[0][1]:.3f} → {LOSSES[-1][1]:.4f}")
trainer.model.save_pretrained(WORK / "adapter")
tuned = trainer.model
tuned.config.use_cache = True
_ = tuned.eval()

## 5. 学習後評価（学習前と同じ評価セット・同じ生成設定）

**見るところ**

- 種類ごとの正解率の変化。
  - `exact`: 学習した質問そのもの。定義文を覚えたか。
  - `paraphrase`: 言い換え・別名・逆引きに、覚えた定義を使えたか。質問の言い回しでしか引き出せないのか。
  - `unknown`: 用語集にない語に、学習した別の用語の定義や一般知識を当てはめて断定していないか（演習のセクション 6 と同じ論点）。データ A には負例が無く、データ B には負例があります。
- `退行`: 学習前は ○ だったのに学習後に × になった質問。
- 出力の原文。定義文が別の用語と混ざっていないか、途中で崩れていないか。

In [ ]:
after_summary, after_rows = evaluate(tuned, "after")
display(pd.DataFrame([before_summary, after_summary]).set_index("label")[["n", "overall", *KINDS, "seconds"]])
regressed = [b["q"] for b, a in zip(before_rows, after_rows) if b["ok"] and not a["ok"]]
print("退行:", regressed)
display(pd.DataFrame([{"kind": b["kind"], "質問": b["q"], "学習前の出力": b["raw"], "前": mark(b["ok"]), "学習後の出力": a["raw"], "後": mark(a["ok"])}
                      for b, a in zip(before_rows, after_rows)]))

## 6. 比較: 学習せずに用語集を検索してプロンプトに入れる

学習の代わりに、質問に含まれる用語名・別名で用語集を検索し、見つかった項目（最大 2 件）を **学習前のモデル**（adapter を無効にした `tuned`）のプロンプトに入れて答えさせます。第5回で扱う「用語集の検索ツール」を、ごく単純にしたものです。

- 検索は文字列の一致だけです（1 文字の別名は除き、英字の語は単語の区切りで一致させる）。用語名を含まない逆引きの質問では、検索に失敗します。
- 該当なしのときは「（該当する項目はありません）」と渡します。

**見るところ**

- 学習後（セクション 5）と比べて、種類ごとの正解率はどうか。
- `検索ヒット`: 根拠の項目を取り出せたか。取り出せなかった質問はどうなったか。
- `unknown` の語に、似た名前の項目（例: 「トークンブリッジ」→ トークン）が引っかかったときにどう答えたか。

In [ ]:
TERMS = json.loads(GLOSSARY_FILE.read_text(encoding="utf-8"))["terms"]

def name_in(name, q):
    if name.isascii():   # 英字の語は単語の区切りで一致させる（「PR」が「PRompt」に当たらないように）
        return re.search(rf"(?<![A-Za-z]){re.escape(name)}(?![A-Za-z])", q, re.I) is not None
    return name in q

def retrieve(q, k=2):
    scored = []
    for t in TERMS:
        names = [n for n in [t["term"], *t["aliases"]] if len(n) >= 2]
        matched = [len(n) for n in names if name_in(n, q)]
        if matched: scored.append((max(matched), t))
    scored.sort(key=lambda x: -x[0])   # 長い名前で一致した項目を優先する
    return [t for _, t in scored[:k]]

with tuned.disable_adapter():
    rag_summary, rag_rows = evaluate(tuned, "retrieval", retrieve=retrieve)

display(pd.DataFrame([before_summary, after_summary, rag_summary]).set_index("label")[["n", "overall", *KINDS, "seconds"]])
display(pd.DataFrame([{"kind": r["kind"], "質問": r["q"], "検索ヒット": ", ".join(r["retrieved"]) or "（なし）",
                       "正しい項目": it["term_id"] or "（用語集にない）", "検索方式の出力": r["raw"], "判定": mark(r["ok"])}
                      for it, r in zip(EVAL_ITEMS, rag_rows)]))

## 7. 結果のまとめ

設定、版、学習時間、loss、学習前・学習後・検索方式の要約を `result.json` にまとめます（`WORK` に保存）。データ A と B の両方を実行した場合は、`out/glossary_sft/` と `out/glossary_sft_plus/` の `result.json` を見比べます。

In [ ]:
result = {
    "config": CONFIG,
    "versions": VERSIONS,
    "environment": "Colab" if IN_COLAB else "ローカル",
    "device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else DEVICE,
    "method": "LoRA（量子化なし）",
    "train_rows": len(train_rows), "train_negative_rows": n_negative,
    "train_seconds": TRAIN_SECONDS, "train_steps": trainer.state.global_step, "loss_first": LOSSES[0][1], "loss_last": LOSSES[-1][1],
    "before": before_summary, "after": after_summary, "retrieval": rag_summary,
    "regressed": regressed,
    "executed_training_myself": True,
}
(WORK / "result.json").write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(result, ensure_ascii=False, indent=2))